In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ppmi_utils import (
    cohort_map,
    event_id_to_visit,
    fields,
    load_ppmi_csvs,
    merge_ppmi_tables,
    numeric_fields,
    parse_imaging_protocol,
    plot_bar,
    plot_hist,
    primdiag_map,
    visit_to_event_id,
)

ROOT_DIR = Path(os.getcwd()).resolve().parents[1]
PPMI_CLINICAL = ROOT_DIR / "csv_dir" / "PPMI_CLINICAL"
PPMI_OTHERS = ROOT_DIR / "csv_dir" / "PPMI_OTHERS"
curated_data = PPMI_OTHERS / "PPMI_Curated_Data_Cut_Public_20251112.xlsx"
ppmi_imaging_descriptions_path = "/home/falconnier/Documents/mri-preprocessing/csv_exploration/PPMI_explo/ppmi_imaging_descriptions.json"
ppmi_imaging_ignored_path = "/home/falconnier/Documents/mri-preprocessing/csv_exploration/PPMI_explo/ppmi_imaging_ignored.csv"

# CSV preprocessing

### Uncurated clinical tabular data 

Not cureated, so might not be used unless missing info in the curated data need to be found here

In [ ]:
ppmi_data = load_ppmi_csvs(PPMI_CLINICAL)
print(f"Available dataframes: {list(ppmi_data.keys())}")

In [ ]:
uncurated_clinical_df = merge_ppmi_tables(ppmi_data)
print("Uncurated DataFrame shape:", uncurated_clinical_df.shape)
print("Uncurated DataFrame columns:", uncurated_clinical_df.columns.tolist())

### Official curated merge tabular data

In [ ]:
curated_clinical_df = pd.read_excel(curated_data)
print("Curated Clinical DataFrame shape:", curated_clinical_df.shape)
print("Curated Clinical DataFrame columns:", curated_clinical_df.columns.tolist())

### idaSearch (image info)

In [ ]:
# ida search csv load and preprocessing
ida_df = pd.read_csv(PPMI_OTHERS / "idaSearch_18Feb2026.csv", low_memory=False)
ida_df = ida_df[~ida_df["Modality"].isin(["Path", "CT"])].copy()
ida_df["Weight"] = ida_df["Weight"].replace(0, np.nan)
ida_df["Age"] = ida_df["Age"].replace(0, np.nan)
ida_df["Study Date"] = pd.to_datetime(ida_df["Study Date"], errors="coerce")
ida_df = ida_df.rename(columns={"Subject ID": "PATNO"})
protocol_parsed = ida_df["Imaging Protocol"].apply(parse_imaging_protocol)
for field in fields:
    ida_df[field] = protocol_parsed.apply(lambda x: x.get(field, np.nan))
for field in numeric_fields:
    ida_df[field] = pd.to_numeric(ida_df[field], errors="coerce")

# avanced modalities
with open(ppmi_imaging_descriptions_path, "r") as f:
    modality_dict = json.load(f)


def normalize_list(lst):
    return [s.strip().upper() for s in lst]


dwi_set = set(normalize_list(modality_dict["dwi"]))
func_set = set(normalize_list(modality_dict["func"]))
anat_sets = {k: set(normalize_list(v)) for k, v in modality_dict["anat"].items()}
ignore_df = pd.read_csv(ppmi_imaging_ignored_path)
ignore_set = set(ignore_df["Description"].str.strip().str.upper())
ida_df["Description_clean"] = ida_df["Description"].astype(str).str.strip().str.upper()
ida_df = ida_df[~ida_df["Description_clean"].isin(ignore_set)].copy()


def classify_advanced(row):
    desc = row["Description_clean"]
    modality = row["Modality"]
    # --- DWI ---
    if desc in dwi_set:
        return "DWI"
    # --- Functional ---
    if desc in func_set:
        return "rsfMRI"
    # --- Anatomical ---
    for anat_type, anat_set in anat_sets.items():
        if desc in anat_set:
            return anat_type  # T1w, T2w, T2starw, FLAIR
    # --- Nuclear imaging ---
    if modality == "PET":
        return "PET"
    if modality == "SPECT":
        return "SPECT"
    # --- fallback ---
    if modality == "MRI":
        return "MRI_other"
    if modality == "fMRI":
        return "fMRI_other"
    return "Unknown"


ida_df["Advanced_Modality"] = ida_df.apply(classify_advanced, axis=1)

# print(ida_df["Advanced_Modality"].value_counts())
# print(pd.crosstab(ida_df["Modality"], ida_df["Advanced_Modality"]))
print("ida searchDataFrame shape:", ida_df.shape)
print(ida_df.columns.tolist())

### Merge clinical and imaging info

In [ ]:
# merge clinical and imaging dataframes
ida_df["PATNO"] = ida_df["PATNO"].astype(str)
curated_clinical_df["PATNO"] = curated_clinical_df["PATNO"].astype(str)
ida_df["EVENT_ID"] = ida_df["Visit"].map(visit_to_event_id)
curated_clinical_df["Visit"] = curated_clinical_df["EVENT_ID"].map(event_id_to_visit)
curated_clinical_df["PRIMDIAG_DESC"] = curated_clinical_df["PRIMDIAG"].map(primdiag_map)
curated_clinical_df["COHORT_DESC"] = curated_clinical_df["COHORT"].map(cohort_map)

cohort_map
df = pd.merge(
    curated_clinical_df,
    ida_df,
    # how="left",
    how="outer",
    on=["PATNO", "EVENT_ID"],
    suffixes=("", "_idasearch"),  # Curated stays same, Imaging gets '_ida' suffix
)
print("Final merged DataFrame shape:", df.shape)
print("Final merged DataFrame columns:", df.columns.tolist())

In [ ]:
# patient_ex = df[(df["PATNO"] == "100445") & (df["EVENT_ID"] == "BL")].copy()
# print(patient_ex.shape)
# patient_ex.to_csv("patient_ex.csv", index=False)

# Filters

filter out

In [ ]:
# general filters
print("Shape before filter:", df.shape)
df = df[df["EVENT_ID"].notna()].copy()  # only keep rows with valid event_id
df = df[df["Research Group"] != "Phantom"].copy()  # remove phantom scans
df = df[df["Image ID"].notna()].copy()  # only keep rows with imaging data
print("Shape after filters:", df.shape)

# # filters on images
# df = df[
#     (df["Advanced_Modality"].isin(["T1w", "FLAIR"]))
#     # & (df["Weighting"] == "T1")
#     # & (df["Field Strength"] > 1.4)
#     # & (df["Matrix Z"] > 100)
#     # & (df["Slice Thickness"] < 1.4)
#     & (df["Acquisition Type"] == "3D")
# ].copy()

# print(
#     f"Filtered dataset size: {df.shape[0]} images from {df['PATNO'].nunique()} subjects."
# )
# # save as csv for further exploration
# df.to_csv("ppmi_clinical_imaging_merged_filtered.csv", index=False)

In [ ]:
# print percetage of na values in each column in the following list
col_list = ["PRIMDIAG", "COHORT", "Research Group", "subgroup", "Image ID"]
for col in col_list:
    na_percentage = df[col].isna().mean() * 100
    print(f"{col}: {na_percentage:.2f}% NA values")

# Basic analysis

### Patients and visits without images

In [ ]:
# Visits without image ID
df_no_image_id = df[df["Image ID"].isna()]
n_visits_no_image = df_no_image_id.groupby(["PATNO", "EVENT_ID"]).ngroups
print(f"Number of visits without 'Image ID': {n_visits_no_image}")

total_visits = df.groupby(["PATNO", "EVENT_ID"]).ngroups
print(f"Total number of visits: {total_visits}")
print(
    f"Percentage of visits without 'Image ID': {n_visits_no_image / total_visits:.2%}"
)

In [ ]:
# Number of unique patients with at least one visit without image ID
# Total number of patients
total_patients = df["PATNO"].nunique()
# Number of patients with no images at all
patients_with_image = df.loc[df["Image ID"].notna(), "PATNO"].unique()
patients_no_image = set(df["PATNO"]) - set(patients_with_image)
n_patients_no_image = len(patients_no_image)

print(f"Total patients: {total_patients}")
print(f"Patients with no image at all: {n_patients_no_image}")
print(f"Percentage with no image: {n_patients_no_image / total_patients:.2%}")


### Age at baseline

In [ ]:
baseline_df = df[df["EVENT_ID"] == "BL"].copy()
baseline_unique = baseline_df.drop_duplicates(subset=["PATNO"])  # keep first occurence
print("Baseline age summary (one row per patient):")
print(baseline_unique[["age"]].describe().T)
plot_hist(baseline_unique, "age", title="Age at baseline")

### Sex distribution

In [ ]:
# Count occurrences
sex_counts = baseline_unique["Sex"].value_counts()

# Print table
print("Sex distribution at baseline:")
print(sex_counts.to_frame().T)

# Plot pie chart
plt.figure(figsize=(6, 6))
plt.pie(
    sex_counts,
    labels=sex_counts.index,
    autopct="%1.1f%%",  # show percentage
    startangle=90,  # rotate to start from top
)
plt.title("Sex distribution at baseline")
plt.axis("equal")  # equal aspect ratio ensures pie is circular
plt.show()


### Cohort definition
1. Parkinson’s Disease, i.e., people who have a formal diagnosis of Parkinson’s disease (PD) 
2. Prodromal, i.e., people who are at risk of developing PD based on clinical features, genetic variants or other biomarkers but have not been formally diagnosed 
3. Healthy Controls, i.e., people with no neurologic disorder and no first-degree relative with PD
4. SWEDD (Scan without dopaminergic deficit). This is a small legacy cohort that you may wish to exclude, depending on your research purpose; for more details, see https://www.ppmi-info.org/study-design/study-cohorts#legacy/

In [ ]:
# diagnosis at baseline
print("Diagnosis distribution at baseline:")
cohort_counts = baseline_unique["Research Group"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "Research Group",
    title="Research Group distribution at baseline",
)

In [ ]:
cohort_counts = baseline_unique["COHORT_DESC"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "COHORT_DESC",
    title="Cohort distribution at baseline",
)

### subgroup
Subgroup is derived from various source columns to give a more detailed group assignment than cohort. It can take values of Healthy Control, SWEDD, SWEDD/PD, SWEDD/nonPD, Hyposmia, RBD, Sporadic PD, LRRK2, GBA, PINK1, PRKN, SNCA or combinations of genetic variants and/or RBD (e.g. LRRK2 + GBA, GBA + RBD).

In [ ]:
cohort_counts = baseline_unique["subgroup"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "subgroup",
    title="Subgroups distribution at baseline",
)

### PRIMDIAG

In [ ]:
cohort_counts = baseline_unique["PRIMDIAG_DESC"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "PRIMDIAG_DESC",
    title="PRIMDIAG distribution at baseline",
)

# Longitudinal analysis

### Cumulative number of visits through years

In [ ]:
# Ensure Study Date is datetime
df["Study Date"] = pd.to_datetime(df["Study Date"], errors="coerce")

# Keep only unique patient-event pairs
unique_visits = df.drop_duplicates(subset=["PATNO", "EVENT_ID"])
unique_visits = unique_visits[unique_visits["Study Date"] >= "2010-01-01"]
unique_visits["YearMonth"] = unique_visits["Study Date"].dt.to_period("M")
visits_per_month = unique_visits.groupby("YearMonth").size()

# Plot
plt.figure(figsize=(12, 6))
plt.plot(visits_per_month.index.to_timestamp(), visits_per_month.values, marker="o")
plt.title("Number of visits per month (all events)")
plt.xlabel("Month")
plt.ylabel("Number of unique visits")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### How many visits per patient ?

In [ ]:
visits_per_patient = df.groupby("PATNO")["EVENT_ID"].nunique()
print("Visits per patient (summary):")
print(visits_per_patient.describe().to_frame().T)
# Histogram of visit counts
plot_hist(
    visits_per_patient.reset_index(), "EVENT_ID", title="Number of visits per patient"
)

### Longitudinal diagnosis changes

In [ ]:
column_name = "PRIMDIAG_DESC"
# Assuming COHORT_DEFINITION reflects diagnosis
diagnosis_over_time = (
    df.groupby("PATNO")[
        column_name
    ].nunique()  # number of distinct diagnoses per patient
)

# Patients with >1 diagnosis
patients_changed_diag = diagnosis_over_time[diagnosis_over_time > 1]
print(f"Number of patients with diagnosis change: {len(patients_changed_diag)}")

# Optional: show which patients and their diagnoses
diagnosis_per_patient = df.groupby("PATNO")[column_name].unique()
for pat in patients_changed_diag.index:
    print(f"{pat}: {diagnosis_per_patient[pat]}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count of distinct diagnoses per patient
diagnosis_over_time = df.groupby("PATNO")[column_name].nunique()

# Patients with diagnosis changes (>1)
patients_changed_diag = diagnosis_over_time[diagnosis_over_time > 2]

# Plot distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=diagnosis_over_time, color="skyblue")
plt.title("Number of distinct diagnoses per patient")
plt.xlabel("Distinct diagnoses count")
plt.ylabel("Number of patients")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.show()

# Optional: list patients with changed diagnosis
print(f"Number of patients with diagnosis change: {len(patients_changed_diag)}")
for pat in patients_changed_diag.index:
    print(f"{pat}: {df.groupby('PATNO')[column_name].unique()[pat]}")


# Image analysis